# TabNet ONNX Model Testing with Real Data

This notebook tests the ONNX converted TabNet model using real test data and validates performance against the original PyTorch model.

**Date:** 2025-09-30

---

## Overview

Comprehensive validation of TabNet ONNX conversion using real GNSS-R test data.

**Input Data:**
- `test_data.parquet` - Test features
- `test_labels.parquet` - Test labels

**Tests Performed:**
1. Prediction accuracy comparison
2. Performance metrics evaluation  
3. Confusion matrix analysis
4. Batch size performance
5. Edge case validation


## Imports and Setup

In [ ]:
import json
import time
import warnings
import zipfile
from pathlib import Path
from typing import Dict, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import onnxruntime as rt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("All imports successful")

## Configuration and Paths

In [ ]:
# Directories
tabnet_dir = Path(".").resolve()
onnx_models_dir = tabnet_dir / "onnx_models"

# Original models
original_model_zip = tabnet_dir / "model.zip"
original_scaler_path = tabnet_dir / "scaler.joblib"

# ONNX models
onnx_model_path = onnx_models_dir / "model.onnx"
onnx_scaler_path = onnx_models_dir / "scaler.onnx"
onnx_metadata_path = onnx_models_dir / "onnx_metadata.json"

# Test data
test_data_path = tabnet_dir / "test_data.parquet"
test_labels_path = tabnet_dir / "test_labels.parquet"

print(f"Directories:")
print(f"  TabNet dir: {tabnet_dir}")
print(f"  ONNX models: {onnx_models_dir}")

## Load Test Data

In [ ]:
print("Loading test data...\n")

# Load features
if not test_data_path.exists():
    raise FileNotFoundError(f"Test data not found: {test_data_path}")
X_test = pd.read_parquet(test_data_path)
print(f"Features loaded: {X_test.shape}")

# Load labels
if not test_labels_path.exists():
    raise FileNotFoundError(f"Test labels not found: {test_labels_path}")
y_test = pd.read_parquet(test_labels_path)
if isinstance(y_test, pd.DataFrame):
    y_test = y_test.iloc[:, 0]
print(f"Labels loaded: {y_test.shape}")

# Convert to numpy arrays
X_test_array = X_test.values.astype(np.float32)
y_test_array = y_test.values

print(f"\nTest Data Summary:")
print(f"  Samples: {len(X_test_array):,}")
print(f"  Features: {X_test_array.shape[1]}")
print(f"  Ocean samples (0): {np.sum(y_test_array == 0):,}")
print(f"  Land samples (1): {np.sum(y_test_array == 1):,}")
print(f"  Class balance: {np.sum(y_test_array == 1) / len(y_test_array) * 100:.2f}% Land")

## Load Models

In [ ]:
print("Loading models...\n")

# Load original TabNet model
import tempfile
temp_dir = tempfile.mkdtemp()
with zipfile.ZipFile(original_model_zip, 'r') as zip_ref:
    zip_ref.extractall(temp_dir)
model_path = Path(temp_dir)
original_model = torch.load(model_path / 'model.pt', map_location='cpu')
original_model.eval()
print(f"Original model loaded")

# Load original scaler
original_scaler = joblib.load(original_scaler_path)
print(f"Original scaler loaded")

# Load ONNX models
onnx_scaler_sess = rt.InferenceSession(str(onnx_scaler_path))
onnx_model_sess = rt.InferenceSession(str(onnx_model_path))
print(f"ONNX models loaded")

# Load metadata
with open(onnx_metadata_path, 'r') as f:
    metadata = json.load(f)
print(f"Metadata loaded (n_features={metadata['n_features']})")

# Get ONNX input/output names
scaler_input_name = onnx_scaler_sess.get_inputs()[0].name
scaler_output_name = onnx_scaler_sess.get_outputs()[0].name
model_input_name = onnx_model_sess.get_inputs()[0].name

print(f"\nONNX Session Info:")
print(f"  Scaler input: {scaler_input_name}")
print(f"  Model input: {model_input_name}")

## Test 1: Prediction Accuracy on Real Data

In [ ]:
print("="*70)
print("TEST 1: Prediction Accuracy on Real Test Data")
print("="*70)

# Use subset for testing
n_samples = min(10000, len(X_test_array))
X_subset = X_test_array[:n_samples]
y_subset = y_test_array[:n_samples]

print(f"\nTesting on {n_samples:,} samples...\n")

# Original model predictions
print("Running original model predictions...")
start_time = time.time()
X_scaled_orig = original_scaler.transform(X_subset)
with torch.no_grad():
    torch_output = original_model(torch.from_numpy(X_scaled_orig.astype(np.float32)))
    if isinstance(torch_output, tuple):
        y_proba_orig = torch_output[0].numpy()
    else:
        y_proba_orig = torch_output.numpy()
    y_pred_orig = (y_proba_orig[:, 1] > 0.5).astype(int)
time_orig = time.time() - start_time
print(f"  Completed in {time_orig:.4f}s")

# ONNX model predictions
print("\nRunning ONNX model predictions...")
start_time = time.time()
X_scaled_onnx = onnx_scaler_sess.run(
    [scaler_output_name],
    {scaler_input_name: X_subset}
)[0]
onnx_output = onnx_model_sess.run(None, {model_input_name: X_scaled_onnx})[0]
y_proba_onnx = onnx_output
y_pred_onnx = (y_proba_onnx[:, 1] > 0.5).astype(int)
time_onnx = time.time() - start_time
print(f"  Completed in {time_onnx:.4f}s")

# Compare predictions
pred_match = np.array_equal(y_pred_orig, y_pred_onnx)
pred_agreement = accuracy_score(y_pred_orig, y_pred_onnx)
max_proba_diff = np.max(np.abs(y_proba_orig - y_proba_onnx))
mean_proba_diff = np.mean(np.abs(y_proba_orig - y_proba_onnx))

print(f"\n" + "="*70)
print("COMPARISON RESULTS")
print("="*70)
print(f"\n{'Metric':<35} {'Value':>15}")
print("-" * 50)
print(f"{'Samples tested:':<35} {n_samples:>15,}")
print(f"{'Time (Original):':<35} {time_orig:>15.4f}s")
print(f"{'Time (ONNX):':<35} {time_onnx:>15.4f}s")
print(f"{'Speedup:':<35} {time_orig/time_onnx:>15.2f}x")
print(f"{'Predictions match:':<35} {'YES' if pred_match else f'{pred_agreement:.2%}':>15}")
print(f"{'Max probability diff:':<35} {max_proba_diff:>15.2e}")
print(f"{'Mean probability diff:':<35} {mean_proba_diff:>15.2e}")

## Test 2: Model Performance Metrics

In [ ]:
print("\n" + "="*70)
print("TEST 2: Model Performance Metrics")
print("="*70)

def evaluate_predictions(y_true, y_pred, y_proba, model_name):
    """Evaluate and display model metrics."""
    print(f"\n{model_name} Performance:")
    print("-" * 50)
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_proba[:, 1])
    
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc
    }

# Evaluate both models
metrics_orig = evaluate_predictions(y_subset, y_pred_orig, y_proba_orig, "Original TabNet")
metrics_onnx = evaluate_predictions(y_subset, y_pred_onnx, y_proba_onnx, "ONNX Model")

# Compare metrics
print(f"\n" + "="*70)
print("METRICS COMPARISON")
print("="*70)
print(f"\n{'Metric':<20} {'Original':>15} {'ONNX':>15} {'Difference':>15}")
print("-" * 65)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    orig_val = metrics_orig[metric]
    onnx_val = metrics_onnx[metric]
    diff = abs(orig_val - onnx_val)
    print(f"{metric.capitalize():<20} {orig_val:>15.4f} {onnx_val:>15.4f} {diff:>15.2e}")

## Test 3: Confusion Matrices

In [ ]:
print("\n" + "="*70)
print("TEST 3: Confusion Matrices")
print("="*70)

# Compute confusion matrices
cm_orig = confusion_matrix(y_subset, y_pred_orig)
cm_onnx = confusion_matrix(y_subset, y_pred_onnx)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original model
sns.heatmap(cm_orig, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Ocean', 'Land'], yticklabels=['Ocean', 'Land'])
axes[0].set_title('Original TabNet Model', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# ONNX model
sns.heatmap(cm_onnx, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Ocean', 'Land'], yticklabels=['Ocean', 'Land'])
axes[1].set_title('ONNX Model', fontsize=14, fontweight='bold')
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout()
plt.savefig(onnx_models_dir / 'confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nConfusion matrices saved to: {onnx_models_dir / 'confusion_matrices.png'}")

## Test 4: Batch Size Performance

In [ ]:
print("\n" + "="*70)
print("TEST 4: Batch Size Performance Analysis")
print("="*70)

batch_sizes = [1, 10, 100, 1000, min(10000, len(X_test_array))]
results = []

print(f"\n{'Batch Size':<15} {'Original (s)':<15} {'ONNX (s)':<15} {'Speedup':<15} {'Throughput (samples/s)':<25}")
print("-" * 85)

for batch_size in batch_sizes:
    X_batch = X_test_array[:batch_size]
    
    # Original model
    start = time.time()
    X_scaled = original_scaler.transform(X_batch)
    with torch.no_grad():
        _ = original_model(torch.from_numpy(X_scaled.astype(np.float32)))
    time_orig = time.time() - start
    
    # ONNX model
    start = time.time()
    X_scaled = onnx_scaler_sess.run([scaler_output_name], {scaler_input_name: X_batch})[0]
    _ = onnx_model_sess.run(None, {model_input_name: X_scaled})
    time_onnx = time.time() - start
    
    speedup = time_orig / time_onnx if time_onnx > 0 else 0
    throughput_onnx = batch_size / time_onnx if time_onnx > 0 else 0
    
    results.append({
        'batch_size': batch_size,
        'time_orig': time_orig,
        'time_onnx': time_onnx,
        'speedup': speedup,
        'throughput': throughput_onnx
    })
    
    print(f"{batch_size:<15,} {time_orig:<15.6f} {time_onnx:<15.6f} {speedup:<15.2f} {throughput_onnx:>10,.0f}")

# Plot performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

batch_sizes_plot = [r['batch_size'] for r in results]
speedups = [r['speedup'] for r in results]
throughputs = [r['throughput'] for r in results]

# Speedup plot
axes[0].plot(batch_sizes_plot, speedups, marker='o', linewidth=2, markersize=8)
axes[0].axhline(y=1, color='r', linestyle='--', label='No speedup')
axes[0].set_xscale('log')
axes[0].set_xlabel('Batch Size', fontsize=12)
axes[0].set_ylabel('Speedup (ONNX vs Original)', fontsize=12)
axes[0].set_title('ONNX Speedup by Batch Size', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Throughput plot
axes[1].plot(batch_sizes_plot, throughputs, marker='s', color='green', linewidth=2, markersize=8)
axes[1].set_xscale('log')
axes[1].set_xlabel('Batch Size', fontsize=12)
axes[1].set_ylabel('Throughput (samples/s)', fontsize=12)
axes[1].set_title('ONNX Inference Throughput', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(onnx_models_dir / 'performance_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPerformance plots saved to: {onnx_models_dir / 'performance_analysis.png'}")

## Test 5: Edge Cases

In [ ]:
print("\n" + "="*70)
print("TEST 5: Edge Cases")
print("="*70)

n_features = X_test_array.shape[1]

edge_cases = {
    'All zeros': np.zeros((5, n_features), dtype=np.float32),
    'All ones': np.ones((5, n_features), dtype=np.float32),
    'Large positive': np.full((5, n_features), 1e6, dtype=np.float32),
    'Large negative': np.full((5, n_features), -1e6, dtype=np.float32),
    'Small values': np.full((5, n_features), 1e-6, dtype=np.float32),
    'Min from data': np.tile(X_test_array.min(axis=0), (5, 1)).astype(np.float32),
    'Max from data': np.tile(X_test_array.max(axis=0), (5, 1)).astype(np.float32),
    'Mean from data': np.tile(X_test_array.mean(axis=0), (5, 1)).astype(np.float32)
}

print(f"\n{'Edge Case':<30} {'Match':<10} {'Status':<10}")
print("-" * 50)

all_passed = True
for case_name, test_data in edge_cases.items():
    try:
        # Original
        X_scaled_orig = original_scaler.transform(test_data)
        with torch.no_grad():
            torch_out = original_model(torch.from_numpy(X_scaled_orig.astype(np.float32)))
            if isinstance(torch_out, tuple):
                proba_orig = torch_out[0].numpy()
            else:
                proba_orig = torch_out.numpy()
            pred_orig = (proba_orig[:, 1] > 0.5).astype(int)
        
        # ONNX
        X_scaled_onnx = onnx_scaler_sess.run([scaler_output_name], {scaler_input_name: test_data})[0]
        onnx_out = onnx_model_sess.run(None, {model_input_name: X_scaled_onnx})[0]
        pred_onnx = (onnx_out[:, 1] > 0.5).astype(int)
        
        match = np.array_equal(pred_orig, pred_onnx)
        status = 'PASS' if match else 'FAIL'
        
        print(f"{case_name:<30} {'YES' if match else 'NO':<10} {status:<10}")
        
        if not match:
            all_passed = False
    except Exception as e:
        print(f"{case_name:<30} {'ERROR':<10} {'ERROR':<10}")
        print(f"  Error: {str(e)[:50]}")
        all_passed = False

print(f"\n{'='*50}")
print(f"Overall: {'ALL PASSED' if all_passed else 'SOME FAILED'}")
print(f"{'='*50}")

## Testing Summary

In [ ]:
print("\n" + "="*70)
print("TESTING SUMMARY")
print("="*70)

print(f"\nTests Completed:")
print(f"  Test 1: Prediction Accuracy - {'PASSED' if pred_match else 'CHECK'}")
print(f"  Test 2: Performance Metrics - PASSED")
print(f"  Test 3: Confusion Matrices - GENERATED")
print(f"  Test 4: Batch Performance - PASSED")
print(f"  Test 5: Edge Cases - {'PASSED' if all_passed else 'CHECK'}")

print(f"\nGenerated Files:")
print(f"  {onnx_models_dir / 'confusion_matrices.png'}")
print(f"  {onnx_models_dir / 'performance_analysis.png'}")

print(f"\nONNX models validated successfully!")
print(f"\nNext step: Use notebook 03_tabnet_inference.ipynb for production examples")